# WorldSim v2 Dataset Assembly
Assemble v2 data sources, convert them to messages format, and apply curriculum ordering.


## 1. Repo Root & Environment


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "config" / "generation.yaml").exists():
        raise RuntimeError("Cannot find repo root. Run this notebook from the worldsim-training directory.")

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

print(f"Repo root: {REPO_ROOT}")


## 2. Source Inventory


In [ ]:
import json
from collections import Counter

from scripts.common import read_jsonl

source_paths = {
    "v1 train": REPO_ROOT / "data" / "final" / "worldsim-v31-mix-v1" / "train.jsonl",
    "v1 dev": REPO_ROOT / "data" / "final" / "worldsim-v31-mix-v1" / "dev.jsonl",
    "Batch 1 (I-N)": REPO_ROOT / "data" / "validated" / "batch_v2_01_tasks_in" / "passed.jsonl",
    "Batch 2 (G fix)": REPO_ROOT / "data" / "validated" / "batch_v2_02_task_g_fix" / "passed.jsonl",
    "Negative": REPO_ROOT / "data" / "samples" / "negative_examples.jsonl",
    "General KR": REPO_ROOT / "data" / "samples" / "general_korean.jsonl",
}

for label, path in source_paths.items():
    rows = read_jsonl(path)
    counts = Counter(str(row.get("task", "unknown")) for row in rows)
    print(f"\n=== {label}: {len(rows)} rows ===")
    if counts:
        print(dict(sorted(counts.items())))
    else:
        print("(missing or empty)")


## 3. Run Assembly


In [ ]:
from scripts.assemble_v2_dataset import assemble_v2_dataset

assembly_result = assemble_v2_dataset(REPO_ROOT)
assembly_manifest = assembly_result.manifest
assembly_manifest


## 4. Validate Assembled Data


In [ ]:
assembled_train = read_jsonl(assembly_result.train_path)
assembled_dev = read_jsonl(assembly_result.dev_path)

train_hashes = {row.get("output") if isinstance(row.get("output"), str) else json.dumps(row.get("output"), ensure_ascii=False, sort_keys=True) for row in assembled_train}
dev_hashes = {row.get("output") if isinstance(row.get("output"), str) else json.dumps(row.get("output"), ensure_ascii=False, sort_keys=True) for row in assembled_dev}

print({
    "train_rows": len(assembled_train),
    "dev_rows": len(assembled_dev),
    "duplicate_overlap": len(train_hashes & dev_hashes),
    "train_task_counts": dict(sorted(Counter(str(row.get("task", "unknown")) for row in assembled_train).items())),
    "dev_task_counts": dict(sorted(Counter(str(row.get("task", "unknown")) for row in assembled_dev).items())),
})
assert len(train_hashes & dev_hashes) == 0, "Train/dev overlap detected after dedupe"


## 5. Convert to Messages Format


In [ ]:
from scripts.convert_mixed_final_to_training_format import convert_mixed_final_to_training_format

conversion_result = convert_mixed_final_to_training_format(
    repo_root=REPO_ROOT,
    input_train=assembly_result.train_path,
    input_dev=assembly_result.dev_path,
    source_manifest=assembly_result.manifest_path,
    output_dir=REPO_ROOT / "data" / "training" / "worldsim-v2-mix",
    dataset_id="worldsim-v2-mix",
)

{
    "train_output": str(conversion_result.train_output),
    "dev_output": str(conversion_result.dev_output),
    "manifest_path": str(conversion_result.manifest_path),
    "train_count": conversion_result.train_count,
    "dev_count": conversion_result.dev_count,
}


## 6. Apply Curriculum Ordering


In [ ]:
from scripts.curriculum_order import CURRICULUM_STAGES, curriculum_order
from scripts.common import write_jsonl

converted_train_rows = read_jsonl(conversion_result.train_output)
ordered_train_rows = curriculum_order(converted_train_rows, seed=42)
curriculum_output = conversion_result.train_output.with_name("train_curriculum.jsonl")
write_jsonl(curriculum_output, ordered_train_rows)

{
    "curriculum_output": str(curriculum_output),
    "rows": len(ordered_train_rows),
    "stage_sizes": {
        stage: sum(1 for row in ordered_train_rows if row.get("task") in tasks)
        for stage, tasks in CURRICULUM_STAGES.items()
    },
}


## 7. Final Dataset Stats


In [ ]:
stage_counts = {
    stage: Counter(str(row.get("task", "unknown")) for row in ordered_train_rows if row.get("task") in tasks)
    for stage, tasks in CURRICULUM_STAGES.items()
}

print("=== Final Dataset Stats ===")
print({
    "assembled_train": len(assembled_train),
    "assembled_dev": len(assembled_dev),
    "converted_train": len(converted_train_rows),
    "converted_dev": len(read_jsonl(conversion_result.dev_output)),
    "curriculum_train": len(ordered_train_rows),
})
print("\n=== Stage Breakdown ===")
for stage, counts in stage_counts.items():
    print(f"Stage {stage}: {dict(sorted(counts.items()))}")


## 8. Comparison: v1 vs v2


In [ ]:
v1_train_rows = read_jsonl(REPO_ROOT / "data" / "training" / "worldsim-v31-mix-v1" / "train_converted.jsonl")
v1_dev_rows = read_jsonl(REPO_ROOT / "data" / "training" / "worldsim-v31-mix-v1" / "dev_converted.jsonl")

comparison = {
    "v1": {
        "train": len(v1_train_rows),
        "dev": len(v1_dev_rows),
        "total": len(v1_train_rows) + len(v1_dev_rows),
        "task_counts": dict(sorted(Counter(str(row.get("task", "unknown")) for row in v1_train_rows + v1_dev_rows).items())),
    },
    "v2": {
        "train": len(ordered_train_rows),
        "dev": len(read_jsonl(conversion_result.dev_output)),
        "total": len(ordered_train_rows) + len(read_jsonl(conversion_result.dev_output)),
        "task_counts": dict(sorted(Counter(str(row.get("task", "unknown")) for row in ordered_train_rows + read_jsonl(conversion_result.dev_output)).items())),
    },
}
comparison
